# Tunatron: fine-tuning DoRA di Nemotron-3-Nano-4B su Google Colab (RTX PRO 6000)

Lancia `ai-pipeline/train.py` **così com'è**, senza modificarlo, con lo stesso stack con cui gira sul PC:
torch 2.14.1 · Unsloth 2026.11.2 · TRL 1.15.0 · PEFT 0.21.2 · transformers 5.17.0. Questo notebook aggiunge solo l'ambiente e il lancio.

**Prima di cominciare:**
1. Il notebook **clona la repo** (`REPO_URL`, ramo `REPO_REF`): addestra ciò che è su GitHub, non ciò che hai sul PC. Se hai cambiato il codice, fai `git push` prima. Il dataset (`ai-pipeline/data/train.jsonl`, `val.jsonl`) è nella repo.
2. Su Colab: *Runtime → Cambia tipo di runtime → GPU RTX PRO 6000*. Poi *Esegui tutto* (oppure cella per cella).
3. Drive serve solo per i risultati: ~8 GB liberi in `Il mio Drive/Tunatron/`.

**Differenze rispetto al PC (tutte nelle impostazioni, il codice è lo stesso):**
- batch 2 × accumulo 4, come nel piano (sul PC da 16 GB serviva 1 × 8; l'effettivo è lo stesso, 8);
- nessun limite di RAM per il training; la compilazione dei kernel Mamba usa **tutti i core** e riduce i job solo se la RAM libera non basta (circa 4 GB per job nvcc), per non far morire il runtime.

**Cosa produce** (in `Drive/Tunatron/`): `model_bf16/` (modello fuso, ~7,4 GB: servono altrettanti GB liberi su Drive), `adapter/`, `train_colab.log`, `colab_freeze.txt` (versioni installate), `colab_commit.txt` (commit della repo che è stato addestrato).
Dopo il training, la **sezione 10** quantizza `model_bf16` con NVIDIA ModelOpt e salva in Drive anche `model_fp8/` e `model_nvfp4/` (+ `quantize.log`, `colab_freeze_quant.txt`). Può girare nello stesso runtime o in uno nuovo (legge `model_bf16` da Drive).

**Limiti, da sapere:**
- `train.py` non sa riprendere da un checkpoint: se il runtime viene reclamato a metà, si riparte da zero. Se il browser si disconnette ma il runtime resta vivo, il training continua: riesegui la cella delle impostazioni, quella delle funzioni e quella di monitoraggio.
- `quantize.py` non è mai stato eseguito, nemmeno sul PC (ci sono solo i test delle sue parti pure): la prima esecuzione vera è questa. Per una prova veloce metti `QUANT_EXTRA_ARGS = "--num-samples 16 --max-seq-length 1024"`.
- Il notebook non è stato eseguito su Colab dallo sviluppatore: le funzioni di supporto sono state provate sul PC (clone della repo, scaricamento e patch dei sorgenti, parsing del log, controllo del processo), l'installazione e il training no.


## 0. Impostazioni

In [ ]:
# ============================ IMPOSTAZIONI (le uniche da toccare) ============================
REPO_URL = "https://github.com/BlockchainzGurus/tunatron.git"   # repo pubblica: nessuna credenziale
REPO_REF = "master"                            # ramo, tag o hash di commit da addestrare
DRIVE_DIR = "/content/drive/MyDrive/Tunatron"  # qui finiscono i risultati (modello, adapter, log)
WORK = "/content/Tunatron"                     # dove viene clonata la repo: disco locale di Colab (veloce), non Drive
VENV = "/content/venv"
SRC = "/content/mamba_src"                     # sorgenti patchati dei kernel Mamba

BATCH_SIZE = 2      # come il piano (2 x 4). La RTX PRO 6000 ha 96 GB: non serve 1 x 8 come sul PC da 16 GB
GRAD_ACCUM = 4
EXTRA_TRAIN_ARGS = ""   # altri argomenti per train.py, es. "--epochs 1" oppure "--max-steps 20" per una prova breve

# Versioni con cui il training è stato provato sul PC. PIN_VERSIONS=False installa le ultime (come PIN_VERSIONS=0 di scripts/setup_env.sh).
PIN_VERSIONS = True
TORCH_PIN = "2.14.1"
PINS = {"unsloth": "2026.11.2", "unsloth_zoo": "2026.11.2", "trl": "1.15.0", "peft": "0.21.2", "transformers": "5.17.0",
        "accelerate": "1.15.0", "datasets": "4.8.5", "bitsandbytes": "0.50.2"}
EXTRA_PACKAGES = ["pyarrow", "huggingface_hub"]
# Kernel CUDA di Mamba-2: si compilano dai sorgenti (nessun wheel per torch 2.14), con -std=c++20 al posto di c++17.
KERNELS = [("causal-conv1d", "1.7.0"), ("mamba-ssm", "2.3.2.post1")]

# --- Quantizzazione (sezione 10) ---
VENV_QUANT = "/content/venv-quant"             # ambiente separato: ModelOpt fissa transformers < 5.19 (come .venv-quant sul PC)
MODELOPT_SPEC = "nvidia-modelopt[hf]>=0.48.0"  # la ricetta NVFP4 del 4B c'è dalla 0.48.0
QUANT_FORMATS = ("fp8", "nvfp4")               # uno o entrambi
QUANT_BF16_EMBEDDINGS = False                  # True solo se vLLM non carica l'NVFP4: tiene gli embedding in BF16 (vedi README)
QUANT_EXTRA_ARGS = ""                          # altri argomenti per quantize.py, es. "--num-samples 16 --max-seq-length 1024" per una prova


## 1. Funzioni di supporto

In [ ]:
import glob, hashlib, json, os, re, shlex, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

AI = f"{WORK}/ai-pipeline"                    # cartella da cui parte train.py (i suoi percorsi di default sono relativi)
PY = f"{VENV}/bin/python"
PYQ = f"{VENV_QUANT}/bin/python"
LOG = f"{AI}/outputs/train.log"
PIDFILE = f"{AI}/outputs/train.pid"


def sh(cmd, env=None, cwd=None, check=True):
    """Esegue un comando bash mostrandone l'output man mano; con check=True solleva un errore se esce con codice != 0."""
    print("+", cmd if len(cmd) < 400 else cmd[:400] + " ...", flush=True)
    p = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, **(env or {})}, cwd=cwd)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"comando fallito (exit {rc}): {cmd[:200]}")
    return rc


def out(cmd):
    return subprocess.run(["bash", "-c", cmd], capture_output=True, text=True).stdout.strip()


def fetch_sdist(name, version, dest):
    """Scarica da PyPI il sorgente (sdist) di name==version, controlla lo sha256, lo estrae in dest e ritorna la cartella."""
    dest = Path(dest)
    shutil.rmtree(dest, ignore_errors=True)
    dest.mkdir(parents=True)
    with urllib.request.urlopen(f"https://pypi.org/pypi/{name}/{version}/json", timeout=60) as resp:
        meta = json.load(resp)
    sdist = next(u for u in meta["urls"] if u["packagetype"] == "sdist")
    archive = dest / sdist["filename"]
    urllib.request.urlretrieve(sdist["url"], archive)
    if hashlib.sha256(archive.read_bytes()).hexdigest() != sdist["digests"]["sha256"]:
        raise RuntimeError(f"sha256 diverso da quello di PyPI per {archive.name}")
    with tarfile.open(archive) as tf:
        try:
            tf.extractall(dest, filter="data")
        except TypeError:  # Python senza i filtri di tarfile
            tf.extractall(dest)
    return next(p for p in dest.iterdir() if p.is_dir() and (p / "setup.py").exists())


def patch_cxx20(src):
    """torch >= 2.14 richiede C++20, ma i setup.py dei kernel Mamba fissano -std=c++17: stessa correzione fatta sul PC.
    Ritorna quante occorrenze ha sostituito."""
    setup = Path(src) / "setup.py"
    text = setup.read_text()
    n = text.count("-std=c++17")
    if n == 0 and "-std=c++20" not in text:
        raise RuntimeError(f"{setup}: non trovo -std=c++17 da correggere, la struttura del pacchetto è cambiata")
    setup.write_text(text.replace("-std=c++17", "-std=c++20"))
    return n


def build_kernels(py, tag):
    """Compila causal-conv1d e mamba-ssm per il virtualenv di `py`: sorgenti PyPI freschi (patch c++20), tutti i core che la RAM permette.
    Ogni ambiente li compila contro il proprio torch, in una cartella sua (`tag`): nessun oggetto compilato viene riusato."""
    t0 = time.time()
    sh(f"uv pip install --python {py} packaging ninja wheel setuptools")
    build_env = {"MAX_JOBS": str(JOBS), "NVCC_THREADS": "2", "TORCH_CUDA_ARCH_LIST": ARCH,
                 "CAUSAL_CONV1D_FORCE_BUILD": "TRUE", "MAMBA_FORCE_BUILD": "TRUE"}  # FORCE_BUILD: niente wheel precompilati non adatti
    for name, version in KERNELS:
        src = fetch_sdist(name, version, f"{SRC}/{tag}/{name}")
        print(f"{name}=={version}: patch c++17 -> c++20 in {patch_cxx20(src)} punti di {src}/setup.py", flush=True)
        sh(f"uv pip install --python {py} --no-build-isolation '{src}'", env=build_env)
        print(f"{name}: fatto, {(time.time() - t0) / 60:.1f} min dall'inizio", flush=True)


def dir_size(d):
    return sum(f.stat().st_size for f in Path(d).rglob("*") if f.is_file())


def running():
    """True se il processo di train.py lanciato da questo notebook è ancora vivo (uno zombie non conta)."""
    try:
        pid = int(Path(PIDFILE).read_text())
        state = Path(f"/proc/{pid}/stat").read_text().rsplit(")", 1)[1].split()[0]
    except (FileNotFoundError, ValueError, IndexError):
        return False
    return state != "Z"


def log_status(path, tail_bytes=2_000_000):  # le barre della valutazione (402 passi ogni 100) riempiono il log: serve una coda lunga
    """Ultima riga di avanzamento, ultime metriche e righe sospette (Traceback, OOM, NaN) dal log del training."""
    with open(path, "rb") as fh:
        fh.seek(0, os.SEEK_END)
        fh.seek(max(0, fh.tell() - tail_bytes))
        raw = fh.read().decode("utf-8", "replace")
    raw = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", raw)  # le barre tqdm annidate lasciano sequenze ANSI davanti ai dizionari delle metriche
    lines = [s.strip() for s in re.split(r"[\r\n]+", raw) if s.strip()]

    def last_match(rx):
        return next((m[0] for m in (re.search(rx, s) for s in reversed(lines)) if m), "")

    return {
        "bar": last_match(r"\d+%\|.*\d+/\d+ \[.*\]"),
        "loss": last_match(r"\{'loss'[^}]*\}"),
        "eval": last_match(r"\{'eval_loss'[^}]*\}"),
        "errors": [s[:200] for s in lines if re.search(r"Traceback|OutOfMemoryError|out of memory|'nan'", s)][-3:],
    }


## 2. Controllo della macchina
GPU, driver, toolkit CUDA, core e RAM. Si ferma subito se qualcosa non torna.

In [ ]:
smi = out("nvidia-smi --query-gpu=name,driver_version,compute_cap,memory.total --format=csv,noheader")
if not smi:
    raise RuntimeError("Nessuna GPU: Runtime > Cambia tipo di runtime > scegli la RTX PRO 6000.")
GPU_NAME, DRIVER, ARCH, VRAM = [x.strip() for x in smi.splitlines()[0].split(",")]
m = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", out("nvidia-smi"))
CUDA_DRIVER = (int(m[1]), int(m[2])) if m else (0, 0)
if CUDA_DRIVER[0] < 13:
    raise RuntimeError(f"Il driver supporta CUDA {CUDA_DRIVER[0]}.{CUDA_DRIVER[1]}: torch 2.14 per Blackwell richiede CUDA 13.")

nvcc = shutil.which("nvcc") or next(iter(sorted(glob.glob("/usr/local/cuda*/bin/nvcc"), reverse=True)), None)
if not nvcc:
    raise RuntimeError("nvcc non trovato: i kernel Mamba si compilano e serve il toolkit CUDA 13.x (come sul PC).")
m = re.search(r"release (\d+)\.(\d+)", out(f"'{nvcc}' --version"))
NVCC_VER = (int(m[1]), int(m[2])) if m else (0, 0)
if NVCC_VER[0] != CUDA_DRIVER[0]:
    raise RuntimeError(f"nvcc {NVCC_VER[0]}.{NVCC_VER[1]} non combacia con CUDA {CUDA_DRIVER[0]} di torch: serve il toolkit 13.x.")
CUDA_HOME = str(Path(nvcc).resolve().parent.parent)

CPUS = os.cpu_count() or 1
mem_kb = int(re.search(r"MemAvailable:\s+(\d+)", Path("/proc/meminfo").read_text())[1])
MEM_GIB = mem_kb / 2**20
# tutti i core; si scende sotto solo se la RAM libera non basta (~4 GB per job nvcc), altrimenti il kernel muore di OOM
JOBS = max(1, min(CPUS, int(MEM_GIB // 4)))

os.environ.update(
    CUDA_HOME=CUDA_HOME, PATH=f"{CUDA_HOME}/bin:{os.environ['PATH']}", TORCH_CUDA_ARCH_LIST=ARCH,
    PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false", UV_NO_PROGRESS="1",
    LD_LIBRARY_PATH=f"{CUDA_HOME}/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}".rstrip(":"),
)
print(f"GPU            {GPU_NAME} ({VRAM}), capability {ARCH}, driver {DRIVER} (CUDA {CUDA_DRIVER[0]}.{CUDA_DRIVER[1]})")
print(f"nvcc           {NVCC_VER[0]}.{NVCC_VER[1]} in {CUDA_HOME}")
print(f"CPU / RAM      {CPUS} core, {MEM_GIB:.0f} GiB liberi  ->  compilazione con {JOBS} job" + ("" if JOBS == CPUS else "  (limitati dalla RAM)"))
sh("df -h /content | tail -n 1", check=False)


## 3. Codice e dataset da GitHub
Monta Drive (per i risultati), clona la repo e fa il `--dry-run` di `train.py` (conta gli esempi, nessuna GPU).

In [ ]:
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # facoltativo: scaricamenti più veloci
    except Exception:
        print("Secret HF_TOKEN non impostato (facoltativo: il modello è pubblico).")
except ImportError:
    print("(non siamo su Colab: salto il mount di Drive)")

# Clone superficiale (solo REPO_REF, senza storia). Vale anche a cella rieseguita: aggiorna lo stesso clone senza toccare outputs/
# (ignorata da git); se hai modificato a mano file tracciati che entrano in conflitto, git si ferma e non li sovrascrive.
git_env = {"GIT_TERMINAL_PROMPT": "0"}  # una repo non leggibile deve dare errore, non restare in attesa di un nome utente
if not (Path(WORK) / ".git").exists():
    sh(f"git init -q '{WORK}'")
sh(f"git -C '{WORK}' remote set-url origin '{REPO_URL}' 2>/dev/null || git -C '{WORK}' remote add origin '{REPO_URL}'")
sh(f"git -C '{WORK}' fetch -q --depth 1 origin '{REPO_REF}'", env=git_env)
sh(f"git -C '{WORK}' checkout -q --detach FETCH_HEAD")
COMMIT = out(f"git -C '{WORK}' log -1 --format='%H %ad %s' --date=short")
print("commit addestrato:", COMMIT)
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
Path(f"{DRIVE_DIR}/colab_commit.txt").write_text(f"{REPO_URL} {REPO_REF}\n{COMMIT}\n")
# Controllo di dati e parametri con la sola libreria standard (nessuna GPU, nessun import pesante).
sh(f"python3 train.py --dry-run --batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} {EXTRA_TRAIN_ARGS}", cwd=AI)


## 4. Ambiente Python (uv)
Virtualenv separato in `/content/venv`: non tocca i pacchetti preinstallati di Colab. Stesse versioni e stessi passi di `scripts/setup_env.sh` (un test ne confronta le versioni con queste impostazioni).

In [ ]:
sh("pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv --python 3.12 {VENV}")

torch_spec = f"torch=={TORCH_PIN}" if PIN_VERSIONS else "torch"
# --torch-backend=auto sceglie la build CUDA adatta al driver (qui CUDA 13.2), come in scripts/setup_env.sh
sh(f"uv pip install --python {PY} '{torch_spec}' torchvision --torch-backend=auto")
pkgs = [f"{k}=={v}" if PIN_VERSIONS else k for k, v in PINS.items()] + EXTRA_PACKAGES
sh(f"uv pip install --python {PY} " + " ".join(f"'{p}'" for p in pkgs))


## 5. Kernel Mamba (causal-conv1d, mamba-ssm)
Scaricati da PyPI, con `-std=c++17` portato a `c++20` (lo richiede torch 2.14) e compilati con tutti i core.

In [ ]:
build_kernels(PY, "train")


## 6. Verifica dell'ambiente
Versioni, import dei kernel sulla GPU e test di `train.py`/`common.py` sul nuovo stack.

In [ ]:
check = """
import torch, importlib.metadata as md
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
assert torch.cuda.is_available()
import causal_conv1d, mamba_ssm
from mamba_ssm.ops.triton.ssd_combined import mamba_split_conv1d_scan_combined
print("causal-conv1d", md.version("causal-conv1d"), "| mamba-ssm", md.version("mamba-ssm"))
import unsloth, trl, peft, transformers
print("unsloth", md.version("unsloth"), "| trl", trl.__version__, "| peft", peft.__version__, "| transformers", transformers.__version__)
"""
Path("/content/check_env.py").write_text(check)
sh(f"{PY} /content/check_env.py")
sh(f"uv pip freeze --python {PY} > '{DRIVE_DIR}/colab_freeze.txt'")
# I test di train.py e common.py (loss fusa, correzione DoRA...) sul nuovo stack, senza GPU.
sh(f"CUDA_VISIBLE_DEVICES= PYTHONPATH=. {PY} -B -m unittest tests.test_train tests.test_common", cwd=AI)


## 7. Avvia il training
In background, con il log in `ai-pipeline/outputs/train.log`.

In [ ]:
Path(f"{AI}/outputs").mkdir(parents=True, exist_ok=True)
if running():
    print("Il training è già in corso: salto il lancio. Passa alla cella di monitoraggio.")
else:
    args = [PY, "train.py", "--batch-size", str(BATCH_SIZE), "--grad-accum", str(GRAD_ACCUM), *shlex.split(EXTRA_TRAIN_ARGS)]
    print("+", " ".join(args), f"   (cwd {AI})")
    with open(LOG, "wb") as log:  # il processo vive in una sessione propria: sopravvive all'interruzione della cella
        proc = subprocess.Popen(args, cwd=AI, stdout=log, stderr=subprocess.STDOUT, start_new_session=True, env=os.environ.copy())
    Path(PIDFILE).write_text(str(proc.pid))
    print(f"avviato, pid {proc.pid}. Il log è {LOG}")


## 8. Segui l'avanzamento
Una riga al minuto; copia il log su Drive. Si può interrompere la cella e rieseguirla: il training non si ferma.

In [ ]:
def report():
    st = log_status(LOG)
    print(time.strftime("%H:%M:%S"), st["bar"], "|", st["loss"], "|", st["eval"], flush=True)
    for e in st["errors"]:
        print("  !!", e, flush=True)


Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
while running():
    report()
    shutil.copy(LOG, f"{DRIVE_DIR}/train_colab.log")  # copia sempre aggiornata anche se il runtime sparisce
    time.sleep(60)
report()
shutil.copy(LOG, f"{DRIVE_DIR}/train_colab.log")
text = Path(LOG).read_text(errors="replace")
if "Modello BF16 fuso salvato in" in text:
    print("\nTraining concluso e modello fuso salvato. Passa alla cella che copia i risultati su Drive.")
else:
    print("\nIl processo è terminato SENZA salvare il modello: guarda gli ultimi messaggi qui sotto.\n")
    print(text[-3000:])


## 9. Copia i risultati su Drive

In [ ]:
model_src = Path(AI) / "model_bf16"
if not list(model_src.glob("*.safetensors")):
    raise FileNotFoundError(f"{model_src} non contiene il modello: il training è finito correttamente?")
for src, name in ((model_src, "model_bf16"), (Path(AI) / "outputs" / "adapter", "adapter")):
    dst = Path(DRIVE_DIR) / name
    shutil.copytree(src, dst, dirs_exist_ok=True)
    assert dir_size(src) == dir_size(dst), f"copia di {name} incompleta: {dir_size(src)} != {dir_size(dst)} byte"
    print(f"{name}: {dir_size(dst) / 2**30:.2f} GiB copiati in {dst}")
shutil.copy(LOG, f"{DRIVE_DIR}/train_colab.log")


## 10. Quantizzazione FP8 e NVFP4 (NVIDIA ModelOpt)
Ambiente separato `/content/venv-quant` (ModelOpt fissa `transformers < 5.19`), con lo stesso torch del training e i kernel Mamba ricompilati per quel torch. Poi `quantize.py` produce `model_fp8/` e `model_nvfp4/` dal `model_bf16/` fuso (locale, o copiato da Drive se il runtime è nuovo). In un runtime nuovo esegui prima le celle 0-3 (impostazioni, funzioni, macchina, codice), poi questa sezione. Serve la GPU libera: non lanciarla durante il training.

### 10.1 Ambiente di quantizzazione

In [ ]:
if not Path(PYQ).exists():
    sh(f"uv venv --python 3.12 {VENV_QUANT}")
torch_spec = f"torch=={TORCH_PIN}" if PIN_VERSIONS else "torch"
# Stesso torch del training, così i kernel Mamba già provati con questa versione valgono anche qui: ModelOpt chiede solo torch >= 2.8 e non lo cambia.
sh(f"uv pip install --python {PYQ} '{torch_spec}' --torch-backend=auto")
# [hf] porta transformers < 5.19, accelerate, peft, datasets e deepspeed (si installa senza compilare nulla).
sh(f"uv pip install --python {PYQ} '{MODELOPT_SPEC}' --torch-backend=auto")
build_kernels(PYQ, "quant")  # la calibrazione fa 512 forward da fino a 4096 token: con i kernel di ripiego in PyTorch sarebbe lentissima


### 10.2 Verifica
Import dei kernel e di ModelOpt, ricette NVIDIA caricabili, test di `quantize.py` (senza GPU).

In [ ]:
check = """
import importlib.metadata as md, torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0))
assert torch.cuda.is_available()
import causal_conv1d, mamba_ssm
from mamba_ssm.ops.triton.ssd_combined import mamba_split_conv1d_scan_combined
import transformers, modelopt
import modelopt.torch.quantization as mtq
from modelopt.recipe import load_recipe
from modelopt.torch.export import export_hf_checkpoint
import quantize
quantize.check_modelopt_version(md.version("nvidia-modelopt"))
print("nvidia-modelopt", md.version("nvidia-modelopt"), "| transformers", transformers.__version__, "| mamba-ssm", md.version("mamba-ssm"))
for recipe in (quantize.NVFP4_RECIPE, quantize.FP8_RECIPE):  # le ricette che quantize.py caricherà davvero
    print("ricetta", recipe, "->", len(load_recipe(recipe).quantize.model_dump()["quant_cfg"]), "regole")
"""
Path("/content/check_quant.py").write_text(check)
sh(f"PYTHONPATH=. {PYQ} /content/check_quant.py", cwd=AI)
sh(f"uv pip freeze --python {PYQ} > '{DRIVE_DIR}/colab_freeze_quant.txt'")
sh(f"CUDA_VISIBLE_DEVICES= PYTHONPATH=. {PYQ} -B -m unittest tests.test_quantize tests.test_common", cwd=AI)


### 10.3 Quantizza
Un formato alla volta, GPU liberata fra uno e l'altro; il log resta in `ai-pipeline/outputs/quantize.log`.

In [ ]:
if running():
    raise RuntimeError("Il training è ancora in corso: la quantizzazione ha bisogno della GPU, aspetta che finisca.")
if not QUANT_FORMATS or set(QUANT_FORMATS) - {"fp8", "nvfp4"}:
    raise ValueError(f"QUANT_FORMATS può contenere solo 'fp8' e 'nvfp4', non {QUANT_FORMATS!r}")

model_dir = Path(AI) / "model_bf16"
if not list(model_dir.glob("*.safetensors")):  # runtime nuovo: il modello fuso è solo su Drive
    drive_model = Path(DRIVE_DIR) / "model_bf16"
    if not list(drive_model.glob("*.safetensors")):
        raise FileNotFoundError(f"model_bf16 non c'è né in {model_dir} né in {drive_model}: fai prima il training (sezioni 7-9).")
    print(f"model_bf16 non è sul disco locale: lo copio da {drive_model} ...", flush=True)
    shutil.copytree(drive_model, model_dir, dirs_exist_ok=True)
    assert dir_size(drive_model) == dir_size(model_dir), "copia di model_bf16 da Drive incompleta"

args = [PYQ, "quantize.py", "--model", "model_bf16"]
if len(set(QUANT_FORMATS)) == 1:
    args += ["--only", QUANT_FORMATS[0]]
if QUANT_BF16_EMBEDDINGS:
    args.append("--bf16-embeddings")
args += shlex.split(QUANT_EXTRA_ARGS)
Path(f"{AI}/outputs").mkdir(parents=True, exist_ok=True)
t0 = time.time()
sh("set -o pipefail; " + shlex.join(args) + f" 2>&1 | tee '{AI}/outputs/quantize.log'", cwd=AI)  # pipefail: l'errore di quantize.py non si perde nel tee
print(f"\nquantizzazione finita in {(time.time() - t0) / 60:.1f} min")


### 10.4 Copia i modelli quantizzati su Drive

In [ ]:
for fmt in dict.fromkeys(QUANT_FORMATS):
    src = Path(AI) / f"model_{fmt}"
    if not (src / "hf_quant_config.json").exists() or not list(src.glob("*.safetensors")):
        raise FileNotFoundError(f"{src} non contiene un checkpoint quantizzato: la cella di quantizzazione è finita senza errori?")
    dst = Path(DRIVE_DIR) / src.name
    shutil.copytree(src, dst, dirs_exist_ok=True)
    assert dir_size(src) == dir_size(dst), (f"copia di {src.name} incompleta o con file vecchi in {dst} ({dir_size(src)} != {dir_size(dst)} byte): "
                                            "se su Drive c'è l'esito di una prova precedente, cancella quella cartella e riesegui")
    print(f"{src.name}: {dir_size(dst) / 2**30:.2f} GiB copiati in {dst}")
shutil.copy(f"{AI}/outputs/quantize.log", f"{DRIVE_DIR}/quantize.log")


## 11. Rilascia il runtime (facoltativo)

In [ ]:
# Facoltativo, SOLO dopo aver verificato la copia: disconnette e rilascia il runtime (smette di consumare unità di calcolo).
from google.colab import runtime
runtime.unassign()
